# Individual Database Profiling Report: `TMP_DF8`


--- 
## 1. Introduction

This document serves as the standardized Data Profiling Report for the TMP_DF8 database, a core component of the Teotihuacan Mapping Project's (TMP) legacy data archive. This analysis is situated within Phase 1 of the Digital TMP project, a foundational stage dedicated to the systematic, quantitative evaluation of legacy database architectures. The core purpose of this report is to provide a deep-dive analysis of the TMP_DF8 schema by visualizing and interpreting a comprehensive suite of pre-computed metrics. By presenting this granular, empirical evidence in a structured and reproducible format, this report establishes a baseline understanding of a single database's structural complexity, data quality, and analytical performance.

This report is a visualization and interpretation layer for a standardized set of metrics generated by an automated data profiling pipeline, as defined in Workflow 4 of the Phase 1 plan. It does not represent a live analysis but rather a reproducible summary of the database's state, ensuring that the evaluation is consistent with the analyses of five other legacy and benchmark databases examined in this phase. The analysis will proceed systematically, beginning with a high-level schema overview and complexity assessment, followed by detailed table-level analysis of storage and health metrics, a granular column-level examination of data types and content profiles, and a concluding evaluation of performance on canonical analytical queries.

The findings presented herein for the TMP_DF8 database are not an end in themselves. They serve as a critical piece of foundational evidence that will be synthesized with findings from six parallel reports in a high-level comparative analysis. This comparative study will form the quantitative backbone of the final Phase 1 White Paper, which will present a formal, evidence-based recommendation for a strategic architectural redesign of the unified TMP database in Phase 2.

--- 
## 2. Background


### 2.1. Context and Motivation

The primary goal of Phase 1 of the Digital TMP project is to conduct a systematic and quantitative evaluation of four legacy databases (`TMP_DF8`, `TMP_DF9`, `TMP_DF10`, `TMP_REAN_DF2`) and two modern benchmark databases. As outlined in the project's planning and architectural documents, this evaluation is a foundational step designed to generate the empirical evidence required to inform a strategic architectural redesign in Phase 2. The existing legacy databases, developed over several decades, exhibit a range of structural complexities and performance characteristics that must be rigorously measured and compared before a new, unified schema can be designed.

This report, focused on `TMP_DF8`, is one of six standardized analyses that provide the granular evidence needed for this high-level comparison. By applying a consistent suite of profiling metrics to each database, the project can move beyond anecdotal or theoretical assessments of their respective strengths and weaknesses. This systematic, data-driven approach is critical for justifying the project's final architectural recommendations based on quantitative evidence. The findings from this individual analysis, when synthesized with those from its counterparts, will form the basis of a defensible, evidence-based strategy for building a performant, usable, and maintainable unified database for future Teotihuacan research. 

### 2.2. Data: The `TMP_DF8` Database

The `TMP_DF8` database represents one of the earliest and most foundational comprehensive electronic datasets for the Teotihuacan Mapping Project. Developed between approximately 1975 and 1977 under the direction of George Cowgill, `TMP_DF8` was created to supersede a series of earlier, often incomplete, test files (DF1-DF7). Its primary purpose was to serve as the project's core research database, consolidating data from the intensive surface surveys of the 1960s and 70s into the most complete and accurate digital format available at that time. Unlike the benchmark databases generated for the present analysis, `TMP_DF8` is a true legacy database, originating from a mainframe computing environment (VAX "random access" file system) before being migrated to more modern platforms. This historical context is critical for understanding its unique structure and inherent data quality characteristics.

The content of `TMP_DF8` is a wide-ranging catalog of archaeological and administrative data for 5,050 survey collection tracts or "sites." The database captures an extensive array of variables, including locational coordinates, administrative details of the survey (field workers, collection dates), qualitative site conditions (e.g., vegetation, erosion), architectural interpretations, and, most importantly, quantitative counts of surface artifacts. These artifact counts cover a broad spectrum of material culture, including detailed ceramic tabulations by phase (e.g., Tzacualli, Miccaotli), obsidian tools (blades, points, cores), ground stone implements (manos, metates), figurines, and miscellaneous objects. The scope of `TMP_DF8` was intended to provide a holistic, albeit surface-level, dataset for investigating the urban characteristics of Teotihuacan, from its demographic history to its patterns of craft production and social organization.

The structure of `TMP_DF8` reflects its origins in a pre-relational database era. As documented in the *TMP DB Genealogy v2*, the schema consists of 27 distinct tables, most of which are vertically partitioned segments of what was conceptually a single, wide flat file. The core of the schema is the `ssn_master` table, which holds the primary site identifier (`ssn`). The remaining 26 tables (named `v201`, `v202`, etc.) contain thematic subsets of variables (e.g., architectural features, ceramic counts) and are linked back to the `ssn_master` through the shared `ssn` key. This design, while not fully normalized in a modern sense, represents a form of fragmentation where related attributes for a single site are spread across numerous tables, requiring multiple joins to assemble a complete record.

A defining feature and known issue of the `TMP_DF8` database, as detailed in the *Cowgill (1993) Guide*, is its non-standard handling of missing data. Instead of using the standard SQL `NULL` value to represent the absence of information, `TMP_DF8` employs a convention of sentinel values. In most cases, a value of `-1` is used to indicate "missing data" or "not applicable." For locational coordinates, the value `-999` signals missing data. This design choice has significant negative implications for modern data analysis, as these numeric sentinel values can corrupt statistical calculations (e.g., averages, sums) and force analysts to explicitly filter them out in every query, a practice that is both error-prone and inefficient. This legacy characteristic is a primary example of the technical debt that the Phase 2 redesign seeks to address.


### 2.3. Methods: Database Profiling Metrics

The analysis presented in this report is based on a standardized suite of pre-computed metrics generated by the `02_run_profiling_pipeline.py` script, as outlined in the Phase 1 project plan. This methodological approach ensures that the evaluation of `TMP_DF8` is both reproducible and directly comparable to the analyses of the other five databases under review. The metrics were calculated from a live PostgreSQL instance of the database and saved to disk as discrete JSON and CSV files. This notebook serves as the visualization and interpretation layer for this static, pre-computed data, rather than performing a live analysis. This ensures that the findings reported here are a stable, verifiable snapshot of the database's characteristics at the time the pipeline was executed.

The profiling pipeline gathers data across several distinct categories to provide a holistic assessment of the database. The report will present findings from each of these categories in sequence. **Schema-level metrics** provide a high-level overview, including aggregate object counts (e.g., table count) and total database size. **Table-level metrics** offer a more granular view of individual tables, assessing their size, row counts, and health indicators such as table bloat. **Column-level analysis** provides the deepest insights, examining both the structure (data types, nullability) and content (NULL value percentages, cardinality) of every column. Finally, this report presents custom **interoperability scores** designed to heuristically measure relational complexity, as well as **performance benchmarks** that measure query latency on a set of canonical analytical workloads. This standardized suite of metrics provides the quantitative foundation for the rigorous, evidence-based comparison across all Phase 1 databases.

### 2.4. Hypotheses

Based on the documented history and structure of `TMP_DF8`, this analysis is guided by a central hypothesis regarding its architectural efficiency. As a legacy database originating from a pre-relational, flat-file system, `TMP_DF8` was structured through vertical partitioning, distributing the attributes of a single archaeological site across 26 thematically distinct tables linked by a common `ssn` key. It is hypothesized that this fragmented, quasi-normalized structure, while less complex than the highly normalized schema of `TMP_DF9`, will still impose a significant performance penalty on analytical queries that require joining data across these partitioned tables. Specifically, we predict that the performance benchmark query designed to test join-intensive operations will exhibit a markedly higher latency compared to both the simple baseline scan on a single table within `TMP_DF8` and the equivalent join query on the denormalized benchmark databases. The results of the performance analysis in Section 8 will serve as the primary evidence to either support or refute this hypothesis.

---
## 3. Setup and Configuration


In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import SVG, Markdown, display
from plotly.subplots import make_subplots

# --- CONFIGURATION ---------------------------------------------------
# SET THIS VARIABLE to the name of the database you want to analyze.
# e.g., 'TMP_DF8', 'TMP_DF9', 'tmp_benchmark_wide_numeric', etc.
DATABASE_NAME = "TMP_DF8"  # <--- CHANGE THIS
# ---------------------------------------------------------------------

# --- Path Definitions ---
# Use relative paths from the notebook's location in reports/individual_db_analysis/
METRICS_DIR = Path("../../outputs/metrics")
ERDS_DIR = Path("../../outputs/erds")

# --- Styling and Display Options ---
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


def display_header(title):
    display(Markdown(f"### {title}"))


def load_metric_file(metric_name, file_type="csv"):
    """Helper function to safely load a metric file."""
    file_path = METRICS_DIR / f"{DATABASE_NAME}_{metric_name}.{file_type}"
    if not file_path.exists():
        print(f"⚠️ WARNING: Metric file not found: {file_path.name}")
        return None
    if file_type == "csv":
        return pd.read_csv(file_path)
    elif file_type == "json":
        with open(file_path, "r") as f:
            return json.load(f)


print(f"✅ Setup complete. Analyzing database: '{DATABASE_NAME}'")
print(f"Metrics Directory: {METRICS_DIR}")
print(f"ERD Directory: {ERDS_DIR}")

✅ Setup complete. Analyzing database: 'TMP_DF8'
Metrics Directory: ..\..\outputs\metrics
ERD Directory: ..\..\outputs\erds


---
## 4. Data Loading


In [ ]:
# Load all metric files into variables
basic_metrics = load_metric_file("basic_metrics", "json")
schema_counts = load_metric_file("schema_counts", "json")
interop_metrics = load_metric_file("interop_metrics", "json")

# Load JSON files and convert to DataFrames
table_metrics_data = load_metric_file("table_metrics", "json")
table_metrics_df = pd.DataFrame(table_metrics_data) if table_metrics_data else None

column_structure_data = load_metric_file("column_structure", "json")
column_structure_df = (
    pd.DataFrame(column_structure_data) if column_structure_data else None
)

column_profiles_data = load_metric_file("column_profiles", "json")
column_profiles_df = (
    pd.DataFrame(column_profiles_data) if column_profiles_data else None
)

# Performance benchmarks are still CSV files
performance_df = load_metric_file("performance_benchmarks")

print("✅ Data loading complete.")

✅ Data loading complete.


---
## 5. High-Level Overview & Schema Visualization


### 5.1. Data: Database and Schema-Level Metrics

This section presents a high-level, aggregate analysis of the `TMP_DF8` database, providing a "30,000-foot view" of its overall size, composition, and structural complexity. The metrics presented below are schema-wide statistics, sourced from the `_basic_metrics.json`, `_schema_counts.json`, and `_interop_metrics.json` files generated by the profiling pipeline. These summary statistics offer a quantitative starting point for assessing the database's architecture before proceeding to more granular table and column-level analyses.

The summary table will present five key metrics. `table_count` is a direct measure of structural fragmentation, representing the total number of user-defined tables within the schema. `database_size_mb` quantifies the total disk space consumed by the database, including all tables and indexes. The remaining three are custom heuristic scores designed to measure relational complexity: the **Join Dependency Index (JDI)** measures the density of formal foreign key relationships relative to the number of tables; the **Logical Interoperability Factor (LIF)** assesses the potential for *ad hoc* joins based on column name and data type similarity; and the **Normalization Factor (NF)** provides a composite score that combines other metrics to estimate the overall degree of schema normalization or fragmentation.


### 5.2. Theory & Methods: Schema Complexity and Interoperability Metrics

The schema-level metrics presented in this section are derived from two sources: direct queries against the PostgreSQL `information_schema` catalog and a suite of custom heuristic scores designed to quantify architectural complexity. The combination of these methods provides a multi-faceted, quantitative assessment of the database schema.

**Schema Object Counts:** Fundamental metrics such as `table_count`, `view_count`, and `function_count` are derived from straightforward `COUNT` queries on the relevant tables within the `information_schema`. For example, the table count is obtained by querying `information_schema.tables` where the `table_schema` matches the target schema of the analysis. These direct counts provide a baseline measure of the number of discrete objects that comprise the database structure.

**Interoperability Metrics:** To move beyond simple counts, three custom heuristic metrics were developed to provide a more nuanced assessment of relational complexity. These are defined as follows:
*   **Join Dependency Index (JDI):** The JDI is a measure of relational complexity based on the density of defined foreign key relationships. It is calculated as `JDI = foreign_key_count / max_possible_foreign_keys`, where `max_possible_foreign_keys` is derived from the number of tables (`n`) as `n * (n - 1) / 2`. A JDI score closer to 1.0 indicates a schema with a dense web of explicit relationships, suggesting higher normalization, while a score closer to 0 indicates a schema with few formal relationships, typical of denormalized or fragmented designs.
*   **Logical Interoperability Factor (LIF):** The LIF is a heuristic designed to estimate the potential for *ad hoc* joins where formal foreign keys may not exist. It operates on the assumption that columns with similar names and data types across different tables are likely to represent the same logical entity and are thus joinable. The metric is calculated by counting the number of distinct column name/data type pairs that appear in more than one table, providing a rough measure of logical cohesion.
*   **Normalization Factor (NF):** The NF is a composite score that combines the `table_count` and the `JDI` into a single, normalized value between 0 and 1. This score provides a holistic heuristic for the degree of schema normalization or fragmentation. Higher values suggest a more normalized schema (many tables with dense relationships), while lower values indicate a simpler, denormalized, or fragmented structure. The primary value of these heuristic scores lies not in their absolute values, but in their utility for *relative comparison* across the different database schemas analyzed in Phase 1.


In [ ]:
display_header(f"Key Metrics for: {DATABASE_NAME}")

summary_data = {}
if basic_metrics:
    summary_data.update(basic_metrics)
if schema_counts:
    summary_data.update(schema_counts)
if interop_metrics:
    summary_data.update(interop_metrics)
if table_metrics_df is not None:
    summary_data["total_estimated_rows"] = int(table_metrics_df["row_estimate"].sum())

if summary_data:
    summary_series = pd.Series(summary_data).rename("Value").to_frame()
    display(summary_series)
else:
    print("No summary metrics available.")

### Key Metrics for: TMP_DF8

### 5.3. Results: Key Metrics for `TMP_DF8`

The high-level metrics for `TMP_DF8` provide a clear initial quantitative profile of its architecture. The database has a total of **27 tables** and occupies **20 MB** of disk space. This table count is substantial, immediately indicating a departure from a simple, flat-file structure and suggesting a degree of structural fragmentation.

The custom interoperability metrics offer a more nuanced view of this complexity. The Join Dependency Index (JDI) for `TMP_DF8` is **0.0741**, the Logical Interoperability Factor (LIF) is **2**, and the composite Normalization Factor (NF) is **0.2139**. The relatively low JDI score indicates that despite the high table count, there are few formally defined foreign key relationships between the tables. This is a striking initial finding, as it suggests the schema is fragmented into many tables but lacks the formal relational integrity constraints that would typically accompany such a design in a truly normalized system. This structure is consistent with the database's documented origin as a set of vertically partitioned flat files rather than a natively relational design.


### 5.4. Discussion: Initial Assessment of Schema Complexity

The high-level schema metrics confirm the initial hypothesis that `TMP_DF8` possesses a moderately complex and fragmented structure. The presence of 27 tables for a single logical dataset, combined with a non-trivial Normalization Factor (NF) of 0.2139, provides strong quantitative evidence of a design that divides data into numerous distinct components. This structure is a direct reflection of its documented history as a vertically partitioned system, where thematic groups of variables were stored in separate files.

The most telling metric, however, is the very low Join Dependency Index (JDI) of 0.0741. This score reveals a critical architectural characteristic: while the data is fragmented across many tables, these tables are not linked by formal foreign key constraints. This indicates a schema that is *partitioned* rather than truly *normalized*. The relationships between the tables are logical (based on the shared `ssn` key) but not enforced by the database system itself, placing the burden of maintaining relational integrity entirely on the application or the end user. This lack of formal constraints, coupled with the high table count, creates a schema that is both complex to query and potentially fragile, as it lacks the built-in safeguards of a properly normalized relational database. This initial assessment points to a design that likely impairs analytical usability by requiring manual, multi-table joins without the performance benefits or integrity guarantees of a formally relational system.

### 5.5. Data & Methods: Entity-Relationship Diagram (ERD)

An Entity-Relationship Diagram (ERD) is a visual representation of a database schema that illustrates the entities (tables), their attributes (columns), and the relationships between them. It serves as a critical tool for understanding the logical structure of a database, revealing its degree of normalization, the nature of its data dependencies, and its overall architectural complexity.

The ERD presented in this report was not manually drawn but was generated through an automated process to ensure it provides a completely accurate and objective reflection of the live `TMP_DF8` database schema. The diagram was created by the `03_generate_erds.py` script, which uses the `sqlalchemy-schemadisplay` library to programmatically inspect the live database's metadata. This process automatically discovers all tables, columns, and foreign key constraints and uses the `graphviz` software toolkit to render a graphical representation of these objects and their relationships. This automated methodology guarantees that the ERD is a direct, empirical visualization of the database's actual structure, free from any idealization or interpretation.


In [ ]:
display_header(f"Full ERD for: {DATABASE_NAME}")

try:
    # Find the most recent ERD file for the database
    erd_files = sorted(ERDS_DIR.glob(f"{DATABASE_NAME}_full_ERD_*.svg"), reverse=True)
    if erd_files:
        display(SVG(erd_files[0]))
    else:
        print(f"❌ ERROR: Full ERD SVG file not found for '{DATABASE_NAME}'.")
except Exception as e:
    print(f"An error occurred while displaying the ERD: {e}")

### Full ERD for: TMP_DF8

### 5.6. Results & Discussion: Visualizing Relational Complexity

The Entity-Relationship Diagram for `TMP_DF8` provides powerful visual confirmation of the database's fragmented architecture. The diagram displays a classic "hub-and-spoke" or "starburst" pattern, with the central `ssn_master` table acting as the hub and all 26 other tables (`v201`, `v202`, etc.) radiating outwards as spokes. Each of these peripheral tables is connected back to the central master table via its `ssn` foreign key, but critically, there are no relationships between the peripheral tables themselves. This dense web of tables and radiating connections visually represents the vertical partitioning strategy inherent in `TMP_DF8`'s original design, where a single logical record was physically split across numerous thematic files.

This visual evidence strongly corroborates the quantitative complexity metrics presented earlier. The high `table_count` of 27 is immediately apparent from the sheer number of distinct boxes in the diagram. Furthermore, the ERD perfectly explains the seemingly contradictory combination of a high table count and a low Join Dependency Index (JDI) of 0.0741. While there are many tables, the relationship pattern is simple and repetitive—26 distinct foreign keys all pointing to a single primary key. This results in a low density of relationships relative to the maximum possible number, hence the low JDI. The diagram serves as a clear illustration of a schema that is partitioned rather than deeply relational, providing an undeniable visual testament to the fragmentation that necessitates complex multi-table joins for even basic analytical queries, thereby supporting the central hypothesis about the performance implications of this design.


---
## 6. Table-Level Analysis


### 6.1. Data: Table-Level Metrics

This section transitions from the high-level schema overview to a more granular analysis of the individual tables within the `TMP_DF8` database. The data presented here are sourced from the `TMP_DF8_table_metrics.json` file, which contains a detailed statistical profile for each table. By examining these metrics, we can identify the largest and most resource-intensive tables, assess their general health, and understand how data is distributed across the fragmented schema.

The upcoming summary tables and charts will present four key metrics for each table. The `row_estimate` provides a statistical approximation of the number of rows, offering a quick measure of a table's scale. `column_count` indicates the width or number of attributes in each table. `total_size` reports the total disk space consumed by the table and all of its associated indexes, identifying which tables are the largest contributors to the database's storage footprint. Finally, `bloat_percent` is a critical database health indicator, quantifying the percentage of a table's file that consists of unused, reclaimable space resulting from data modifications, which can impact query performance.


### 6.2. Theory & Methods: Assessing Table Health and Size

The table-level metrics presented in this section are calculated using standard PostgreSQL functions and statistical queries that provide efficient and reliable information about table size and health. The methods used are designed to avoid performance-intensive operations while still yielding accurate assessments.

**Row Estimates:** The `row_estimate` for each table is not derived from an expensive `COUNT(*)` operation, which would require a full table scan. Instead, it is a highly efficient estimate sourced directly from the `pg_class.reltuples` column in PostgreSQL's internal statistics catalog. This value is updated by the `ANALYZE` command (and autovacuum daemon) and typically provides a very close approximation of the actual row count for static or infrequently updated tables, making it a standard and performant method for assessing table size.

**Table and Index Size:** The various size metrics (`table_size`, `index_size`, `total_size`) are calculated using built-in PostgreSQL functions such as `pg_relation_size()` and `pg_total_relation_size()`. These functions measure the actual disk space allocated to the table's data file (the "heap") and its associated indexes. The values are presented in a human-readable format (e.g., kB, MB) for easier interpretation.

**Table Bloat:** Table bloat refers to unused space that accumulates within a PostgreSQL table's data file due to the database's Multi-Version Concurrency Control (MVCC) implementation. When rows are updated (`UPDATE`) or deleted (`DELETE`), the old row versions are not immediately removed from the file; they are marked as "dead" and remain until a `VACUUM` process reclaims the space. `bloat_percent` is a key indicator of database health and is calculated using a standard community-provided SQL query that statistically estimates the amount of this dead space. High bloat percentages can negatively impact performance by increasing the number of disk pages that must be scanned to satisfy a query. It often suggests a need for database maintenance, such as running a `VACUUM FULL` operation or tuning autovacuum settings.


In [ ]:
display_header("Table Metrics Summary")

if table_metrics_df is not None and not table_metrics_df.empty:
    display(
        table_metrics_df.sort_values(
            by="row_estimate", ascending=False
        ).style.background_gradient(
            cmap="viridis", subset=["row_estimate", "bloat_percent"]
        )
    )
else:
    print("No table metrics data available.")

### Table Metrics Summary

### 6.3. Results: Table Metrics Summary for `TMP_DF8`

The table metrics summary for `TMP_DF8` reveals a remarkably consistent and uniform structure across its constituent tables. Of the 27 tables in the database, 25 share an identical row estimate of **5,050 rows**. The two tables with slightly different estimates are `ssn_master` and `v220`, also with an estimate of 5,050, though a more precise row count might differ slightly. This uniformity is a direct and powerful confirmation of the database's vertical partitioning design, where each of these tables represents a thematic slice of the same core set of 5,050 archaeological sites.

In terms of disk usage, the `v401` table, containing locational and high-level site data, is the largest at **512 kB**, followed closely by `v301` at **496 kB**. The remaining tables are generally smaller, with most hovering around 464-472 kB. The bloat percentages are moderately high and consistent across most of the core tables, generally ranging from **44% to 51%**. The `ssn_master`, `v220`, and `v305` tables exhibit the highest bloat at **83.4%**. This suggests that the data may have undergone updates or deletions after its initial load without subsequent `VACUUM` maintenance to reclaim the unused space. However, given the relatively small total size of the database, this level of bloat is unlikely to cause a significant performance impact.


In [ ]:
display_header("Largest Tables by Total Size and Bloat")

if table_metrics_df is not None and not table_metrics_df.empty:
    # Convert pretty size string to bytes for sorting
    def size_to_bytes(s):
        if not isinstance(s, str):
            return 0
        num, unit = s.split()
        num = float(num)
        if "KB" in unit:
            return num * 1024
        if "MB" in unit:
            return num * 1024**2
        if "GB" in unit:
            return num * 1024**3
        return num

    df_copy = table_metrics_df.copy()
    df_copy["total_bytes"] = df_copy["total_size"].apply(size_to_bytes)
    df_copy["bloat_bytes_val"] = df_copy["bloat_bytes"]

    top_10_size = df_copy.nlargest(10, "total_bytes")
    top_10_bloat = df_copy.nlargest(10, "bloat_bytes_val")

    # Display tables
    display(Markdown("**Top 10 Tables by Total Size**"))
    display(
        top_10_size[["table_name", "total_size", "row_estimate"]].reset_index(drop=True)
    )

    display(Markdown("**Top 10 Tables by Bloat Size**"))
    display(
        top_10_bloat[
            ["table_name", "bloat_size", "bloat_percent", "row_estimate"]
        ].reset_index(drop=True)
    )

    # Create subplots
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Top 10 Tables by Total Size", "Top 10 Tables by Bloat Size"),
    )

    fig.add_trace(
        go.Bar(
            y=top_10_size["table_name"],
            x=top_10_size["total_bytes"],
            orientation="h",
            name="Total Size",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            y=top_10_bloat["table_name"],
            x=top_10_bloat["bloat_bytes_val"],
            orientation="h",
            name="Bloat Size",
        ),
        row=1,
        col=2,
    )

    fig.update_layout(
        title_text=f"Table Size Analysis for {DATABASE_NAME}",
        height=500,
        showlegend=False,
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(title_text="Size (Bytes)", row=1, col=1)
    fig.update_xaxes(title_text="Bloat (Bytes)", row=1, col=2)
    fig.show()
else:
    print("No table metrics data available for plotting.")

### Largest Tables by Total Size and Bloat

**Top 10 Tables by Total Size**

**Top 10 Tables by Bloat Size**

### 6.4. Results: Largest Tables by Size and Bloat

The analysis of the largest tables by total size and bloat further highlights the uniform structure of the `TMP_DF8` database. The top 10 tables by total on-disk size are all very similar in scale. The `v401` table is the largest at 512 kB, followed by `v301` at 496 kB, `v303` at 488 kB, and `v213` at 480 kB. The remaining tables in the top 10 are closely clustered between 464 kB and 472 kB. This flat distribution of table sizes corroborates the finding that the data is partitioned relatively evenly across the various thematic tables.

The analysis of table bloat reveals a more distinct pattern. While most of the larger tables exhibit moderate bloat around 0.15 MB, accounting for roughly 45-50% of their total size, three smaller tables—`v220`, `ssn_master`, and `v305`—show exceptionally high bloat percentages of 83.4%. Although the absolute bloat size for these tables is also 0.15 MB, this unused space constitutes a much larger proportion of their smaller total file sizes (344 kB). This pattern suggests that these specific tables, which have fewer columns, may have undergone more significant data modification or deletion relative to their size after the initial data load.

### 6.5. Discussion: Identifying Key Tables and Health Concerns

The table-level analysis provides definitive, quantitative evidence of the `TMP_DF8` database's core architectural principle: vertical partitioning. The consistent row estimate of 5,050 across nearly every table is the most significant finding. This uniformity proves that these tables are not independent entities but are instead fragments of a single, coherent dataset, each holding a different set of attributes for the same 5,050 archaeological sites. The database is effectively a single logical spreadsheet that has been sliced vertically into 27 separate tables. This structure directly informs the performance hypothesis, as any query requiring a comprehensive view of a site will necessitate joining many of these 5,050-row tables back together.

The health metrics indicate a moderate to high level of table bloat across the schema, with several smaller tables reaching over 83%. While this is a potential health concern from a database administration perspective, the total size of the database (20 MB) is very small by modern standards. Consequently, the performance impact of this bloat is likely negligible. The more significant implication is what the bloat suggests about the database's history: the data was likely loaded and then subsequently modified or had records deleted, leaving behind the unused space. For an analytical dataset that is now effectively static, this historical artifact is more of a curiosity than a pressing operational issue. The primary takeaway from the table-level analysis remains the confirmation of the highly partitioned structure, which has direct and significant implications for query complexity and analytical usability.


---
## 7. Column-Level Analysis


### 7.1. Data Type Frequencies

#### 7.1.1. Data & Methods

This analysis examines the fundamental composition of the `TMP_DF8` schema by profiling the data types used across all of its columns. The data for this section is sourced from the `TMP_DF8_column_structure.json` file, which contains metadata for every column in the database, including its assigned PostgreSQL data type. The method involves aggregating this data to count the frequency of each distinct data type. The resulting distribution provides critical insight into the database's design philosophy, particularly regarding how different kinds of information (e.g., categorical, numeric, textual) are physically stored. This choice has direct implications for storage efficiency, data integrity, and analytical usability.


In [ ]:
display_header("Data Type Distribution")

if column_structure_df is not None:
    type_counts = column_structure_df["data_type"].value_counts().reset_index()
    type_counts.columns = ["data_type", "count"]

    # Calculate percentages
    type_counts["percentage"] = (
        type_counts["count"] / type_counts["count"].sum() * 100
    ).round(2)

    # Display comprehensive table
    display(Markdown("**Complete Data Type Distribution**"))
    display(type_counts.style.format({"percentage": "{:.2f}%"}))

    # Display summary statistics
    display(Markdown("**Data Type Summary**"))
    summary_stats = pd.DataFrame(
        {
            "Total Columns": [type_counts["count"].sum()],
            "Unique Data Types": [len(type_counts)],
            "Most Common Type": [
                f"{type_counts.iloc[0]['data_type']} ({type_counts.iloc[0]['count']} columns)"
            ],
            "Least Common Type": [
                f"{type_counts.iloc[-1]['data_type']} ({type_counts.iloc[-1]['count']} columns)"
            ],
        }
    )
    display(summary_stats)

    fig = px.bar(
        type_counts,
        x="data_type",
        y="count",
        title=f"Column Data Type Frequencies in {DATABASE_NAME}",
        labels={"count": "Number of Columns", "data_type": "Data Type"},
    )
    fig.show()
else:
    print("No column structure data available.")

### Data Type Distribution

**Complete Data Type Distribution**

**Data Type Summary**

,Total Columns,Unique Data Types,Most Common Type,Least Common Type
0,320,2,smallint (313 columns),text (7 columns)


#### 7.1.2. Results & Discussion

The analysis of column data types reveals a schema with an overwhelmingly dominant type. Of the 320 columns in the `TMP_DF8` database, **313 (97.81%)** are defined as `smallint` (a two-byte integer). The remaining **7 columns (2.19%)** are defined as `text`. There are no other data types used in the entire schema.

This extreme dominance of the `smallint` data type is a direct reflection of the database's legacy design, which predates the common use of more descriptive types like `VARCHAR` or `BOOLEAN`. The `text` columns are used exclusively for identifiers such as `site`, `subsite`, and `unit`. All other variables, including those representing categorical data, qualitative assessments, and even counts of artifacts, are stored as numeric codes. As documented in the *Cowgill (1993) Guide to Teotihuacan DF8*, these integers are codes that map to specific meanings (e.g., for `qualcnst` (construction quality), a code of '1' means "Very simple or below average," '2' means "Ordinary," etc.).

This design has significant implications for usability. It forces the analyst to constantly refer to external documentation (the codebook) to understand the meaning of the data, making exploratory analysis cumbersome and unintuitive. The use of numeric codes for what is logically categorical data introduces the risk of analytical errors, as these columns could be improperly treated as continuous variables in statistical calculations. This data type profile provides strong evidence for a design philosophy that prioritized storage efficiency in a resource-constrained legacy environment at the cost of the clarity and semantic richness that modern data types provide.

### 7.2. Data Completeness: NULL Value Analysis

#### 7.2.1. Data & Methods

This analysis assesses data completeness and quality by measuring the prevalence of `NULL` values across every column in the `TMP_-DF8` database. Sourced from the `TMP_DF8_column_profiles.json` file, the core metric is `null_percent`, which calculates the percentage of rows containing a `NULL` for each column. In standard SQL, `NULL` is the correct and conventional representation for missing or unknown data. A high percentage of `NULL` values in a column can indicate issues with the original data collection process or subsequent data entry, potentially impacting the reliability of any analysis that relies on that column. This analysis is therefore a critical measure of overall data quality and integrity.


In [ ]:
display_header("Top 20 Columns by Percentage of NULL Values")

if column_profiles_df is not None and not column_profiles_df.empty:
    # Ensure we only show columns with NULLs
    null_df = column_profiles_df[column_profiles_df["null_percent"] > 0].copy()

    if not null_df.empty:
        # Create a full column identifier for clarity
        null_df["full_column_name"] = (
            null_df["tablename"] + "." + null_df["column_name"]
        )

        top_20_nulls = null_df.nlargest(20, "null_percent")

        # Display table
        display(Markdown("**Top 20 Columns with Highest NULL Percentages**"))
        table_display = top_20_nulls[
            [
                "full_column_name",
                "null_percent",
                "null_count_estimate",
                "row_count_exact",
            ]
        ].copy()
        table_display.columns = ["Column", "NULL %", "NULL Count", "Total Rows"]
        display(table_display.reset_index(drop=True))

        # Display summary statistics
        display(Markdown("**NULL Value Summary**"))
        null_summary = pd.DataFrame(
            {
                "Total Columns Analyzed": [len(column_profiles_df)],
                "Columns with NULLs": [len(null_df)],
                "Columns with 100% NULLs": [
                    len(null_df[null_df["null_percent"] == 100])
                ],
                "Average NULL %": [f"{null_df['null_percent'].mean():.2f}%"],
                "Median NULL %": [f"{null_df['null_percent'].median():.2f}%"],
            }
        )
        display(null_summary)

        fig = px.bar(
            top_20_nulls,
            y="full_column_name",
            x="null_percent",
            orientation="h",
            title=f"Top 20 Columns by NULL Percentage in {DATABASE_NAME}",
            labels={
                "null_percent": "Percentage of Rows that are NULL (%)",
                "full_column_name": "Column",
            },
        )
        fig.update_layout(height=600)
        fig.update_yaxes(autorange="reversed")
        fig.show()
    else:
        print("✅ Excellent! No columns with NULL values were found.")

        # Still show summary even when no NULLs
        display(Markdown("**NULL Value Summary**"))
        null_summary = pd.DataFrame(
            {
                "Total Columns Analyzed": [len(column_profiles_df)],
                "Columns with NULLs": [0],
                "Data Completeness": ["100% - Perfect!"],
            }
        )
        display(null_summary)
else:
    print("No column profile data available.")

### Top 20 Columns by Percentage of NULL Values

✅ Excellent! No columns with NULL values were found.


**NULL Value Summary**

,Total Columns Analyzed,Columns with NULLs,Data Completeness
0,320,0,100% - Perfect!


#### 7.2.2. Results & Discussion

The NULL value analysis of `TMP_DF8` yields a striking and highly significant result: out of 320 columns profiled, **zero columns contain any `NULL` values**. The calculated data completeness for the entire database is a perfect 100%.

However, this result is not an indicator of flawless data collection. On the contrary, it is a direct symptom of a non-standard legacy design practice. As extensively documented in the historical project guides, particularly the *Cowgill (1993) Guide to Teotihuacan DF8*, the original database architects opted to use **sentinel values** instead of standard SQL `NULL`s to represent missing or inapplicable data. Throughout the database, the integer `-1` is consistently used to mean "missing data," while the value `-999` is used for missing coordinate information. The "perfect" 100% completeness score is therefore misleading; it masks an unknown quantity of missing data that is encoded as a number.

This design choice has severe negative implications for data integrity and analysis. By encoding missingness as a numeric value, statistical calculations such as averages, sums, or standard deviations will be corrupted unless these sentinel values are explicitly filtered out of every single query. This practice shifts the burden of handling missing data from the database system to the end user, creating a high risk of analytical error if a user is unaware of these "magic numbers." This finding provides a powerful argument for a key recommendation in the Phase 2 redesign: the systematic conversion of all legacy sentinel values to standard SQL `NULL`s to ensure data integrity and improve analytical usability.

### 7.3. Data Complexity: Cardinality Analysis

#### 7.3.1. Data & Methods

This section analyzes the complexity of the data within each column by measuring its **cardinality**. The data is sourced from the `_column_profiles.json` file. Cardinality is defined as the number of unique or distinct values present in a column. This metric is a fundamental characteristic of a dataset and is critical for understanding the nature of each attribute.

The analysis of cardinality helps to distinguish between different types of columns. Columns with very high cardinality, where nearly every value is unique, are typically identifiers or primary keys (e.g., a unique record ID). Conversely, columns with very low cardinality (e.g., 2 to 10 distinct values) usually represent categorical variables, flags, or status codes (e.g., 'Yes'/'No', status types). By examining the distribution of cardinalities across the schema, we can gain insight into the database's structure, identify potential join keys, and understand the complexity of the data that analysts will encounter.


In [ ]:
display_header("Column Cardinality Distribution")

if column_profiles_df is not None and not column_profiles_df.empty:
    # Create a full column identifier
    df = column_profiles_df.copy()
    df["full_column_name"] = df["tablename"] + "." + df["column_name"]

    # Display tables of highest and lowest cardinality columns
    display(Markdown("**Columns with Highest Cardinality (Most Unique)**"))
    display(
        df.nlargest(10, "distinct_values_estimate")[
            ["full_column_name", "distinct_values_estimate"]
        ]
    )

    display(Markdown("**Columns with Lowest Cardinality (Least Unique)**"))
    display(
        df[df["distinct_values_estimate"] > 1].nsmallest(
            10, "distinct_values_estimate"
        )[["full_column_name", "distinct_values_estimate"]]
    )

    # Create a histogram of cardinalities to see the distribution
    fig = px.histogram(
        df,
        x="distinct_values_estimate",
        log_y=True,
        title=f"Distribution of Column Cardinalities in {DATABASE_NAME}",
        labels={"distinct_values_estimate": "Number of Distinct Values (Cardinality)"},
    )
    fig.show()
else:
    print("No column profile data available.")

### Column Cardinality Distribution

**Columns with Highest Cardinality (Most Unique)**

**Columns with Lowest Cardinality (Least Unique)**

#### 7.3.2. Results & Discussion

The cardinality analysis reveals a highly skewed distribution of data complexity across the `TMP_DF8` schema, providing deep insight into its underlying design. As expected, the columns with the highest cardinality are identifiers and raw counts. All `ssn` columns function as unique primary keys, with an estimated 5,050 distinct values corresponding to each archaeological site. Other high-cardinality columns represent quantitative totals that vary widely across the landscape, such as `v401.obsitots` (total obsidian) with 386 unique values, `v302.tlamimil` (Tlamimilolpa phase sherds) with 233, and various coordinate and area measurements.

The most significant finding, however, is the vast number of columns with extremely low cardinality. The output tables and histogram show a long tail of attributes with fewer than 10 unique values. For example, columns like `v207.mcxunitn` and `v212.celts` have only 2 distinct values, while numerous others representing site conditions, architectural features, or specific artifact presence/absence codes (e.g., `v201.midden`, `v207.frstndwl`) have between 3 and 8 unique values. This pattern is a direct, quantitative reflection of the database's fundamental design.

This prevalence of numerous low-cardinality columns is a classic symptom of the "column-based artifact design," a practice identified in the Phase 1 White Paper as a violation of First Normal Form (1NF). Instead of storing artifact counts or categorical observations in a "long" format (e.g., `ssn, artifact_type, count`), the schema uses a separate column for each specific artifact type or observation. This structure is profoundly inefficient for aggregate analysis. To calculate a summary statistic, such as the total number of artifacts from a specific ceramic phase at a site, an analyst must manually identify and sum the values from dozens of disparate, low-cardinality columns. This design choice makes the database cumbersome for modern analytical workflows and provides strong evidence that a structural redesign is necessary to create a more usable and efficient data environment for Phase 2.


---
## 8. Performance Benchmark Analysis


### 8.1. Data, Theory & Methods

This section evaluates the analytical performance of the `TMP_DF8` database by measuring query execution latency on a set of standardized, canonical queries. The data for this analysis is sourced from the `TMP_DF8_performance_benchmarks.csv` file, which logs the results of these benchmark tests. The methodology is designed to provide a fair, reproducible, and representative test of the schema's efficiency under different analytical workloads.

The methodology involves executing a predefined set of three canonical queries against the database and measuring the time taken for each to complete, reported in milliseconds. These queries are not generic; they are hand-crafted and stored in the `phases/01_LegacyDB/sql/canonical_queries/canonical_queries_df8.sql` file to specifically test the architectural characteristics of the `TMP_DF8` schema. The three queries represent distinct analytical workloads:
1.  **Baseline Scan:** A simple `COUNT(*)` on the main site table (`ssn_master`) to establish a baseline for raw I/O performance.
2.  **Multi-Table Join:** A query that joins the location information table (`v401`) with an artifact count table (`v301`) to simulate a typical analytical task requiring data from the vertically partitioned schema.
3.  **Complex Filtering:** A query that joins three tables and applies filtering conditions on attributes from different tables, representing a more complex analytical scenario.

By comparing the latency of the join-intensive queries to the baseline, we can quantitatively measure the performance impact of `TMP_DF8`'s fragmented design, directly testing the central hypothesis of this report.


In [ ]:
display_header("Canonical Query Performance Results")

if performance_df is not None and not performance_df.empty:
    display(performance_df[["query_name", "latency_ms", "status"]])

    # Plot the results for successful queries
    success_df = performance_df[performance_df["status"] == "Success"]
    if not success_df.empty:
        fig = px.bar(
            success_df,
            x="query_name",
            y="latency_ms",
            title=f"Query Latency for {DATABASE_NAME}",
            labels={"latency_ms": "Latency (ms)", "query_name": "Canonical Query"},
        )
        fig.show()
else:
    print("No performance benchmark data available.")

### Canonical Query Performance Results

### 8.2. Results: Query Performance for `TMP_DF8`

The performance benchmark results for `TMP_DF8` provide clear, quantitative data on the impact of its schema design on analytical query latency. The three canonical queries executed successfully, with the following recorded latencies:

*   **Baseline Performance - Query 1.1:** 1.02 ms
*   **Join Performance - Query 2.1:** 6.98 ms
*   **Complex Filtering - Query 3.1:** 8.48 ms

The baseline query, a simple count on a single table, executed almost instantaneously at just over 1 ms, establishing a benchmark for minimal I/O overhead. The join performance query, which required joining two of the core 5,050-row tables (`v401` and `v301`), took **6.98 ms**. This represents a performance degradation of **6.8 times** compared to the baseline scan. The complex filtering query, which involved a three-table join and filtering conditions, was the slowest at **8.48 ms**, representing a performance penalty of **8.3 times** relative to the baseline. These results clearly demonstrate a substantial and measurable increase in query execution time when operations require joining data across the database's vertically partitioned tables.


### 8.3. Discussion: Impact of Schema on Analytical Performance

The performance benchmark results strongly support the central hypothesis that the vertically partitioned schema of `TMP_DF8` imposes a significant performance penalty on analytical queries requiring data integration across multiple tables. The fact that the two-table join query was nearly 7 times slower and the three-table join was over 8 times slower than the simple baseline scan provides direct, quantitative evidence of the overhead introduced by the database's fragmented design.

This performance degradation is a direct and predictable consequence of the architectural choice to store related attributes in separate tables. To fulfill a query that requests information from different thematic categories (e.g., location and artifact counts), the database engine must perform costly join operations, reading and linking data blocks from multiple distinct tables. In contrast, a simple `COUNT(*)` on a single table requires only a sequential scan of one table, resulting in minimal latency. While the absolute query times are low due to the small total size of the database, the *relative* performance degradation is the key finding. It demonstrates that as analytical complexity increases (i.e., as more joins are required), query performance deteriorates substantially. This empirically validates the argument that while partitioning may have been a logical organizational strategy in a flat-file system, it creates an inherent performance bottleneck in a relational query environment. This finding justifies the strategic goal of Phase 2: to create a denormalized schema that minimizes or eliminates the need for such joins to ensure an efficient research environment.

---
## 9. Final Report and Recommendations


### 9.1. Summary of Results

This analysis provides a comprehensive, multi-faceted profile of the `TMP_DF8` legacy database, yielding a set of integrated findings that clearly characterize its architecture, data quality, and performance.

*   **Structural Complexity:**
    The relational structure of `TMP_DF8` is defined by its vertical partitioning. The schema is fragmented into **27 tables**, with 26 thematic tables all linking back to a central `ssn_master` table. This is visually represented in the ERD as a "hub-and-spoke" model. The high table count is offset by a very low Join Dependency Index (JDI) of **0.0741**, indicating a lack of formal relationships beyond the primary links to the central hub. The schema is therefore not deeply relational but is more accurately described as partitioned. The data type usage is extremely homogenous, with **97.8%** of the 320 columns being of the `smallint` type. This reflects a legacy design philosophy of using numeric codes for categorical data, which, while space-efficient, severely impairs usability and requires constant reference to external documentation.

*   **Data Quality & Health Concerns:**
    A defining data quality characteristic of `TMP_DF8` is its non-standard handling of missing data. The analysis found **zero NULL values** across the entire database, a result of the documented practice of using sentinel values (`-1` or `-999`) to represent missingness. This practice presents a significant data integrity risk, as it can corrupt statistical calculations if not explicitly handled by the user in every query. In terms of database health, a moderate to high level of table bloat was observed, with most core tables exhibiting bloat percentages between **44% and 51%**, and three smaller tables reaching **83.4%**. Given the database's small total size (20 MB), this bloat is not a critical performance issue but indicates a history of data modification without subsequent maintenance.

*   **Performance Profile:**
    The performance benchmarks quantitatively confirmed the negative impact of the database's partitioned structure on analytical queries. While a baseline scan of a single table was nearly instantaneous (**1.02 ms**), queries requiring joins across two or three tables were significantly slower, with latencies of **6.98 ms** and **8.48 ms**, respectively. This represents a performance degradation of approximately **7 to 8 times** for common analytical tasks that require integrating data from different thematic tables. These results provide direct, empirical evidence that the schema's fragmentation creates a significant and measurable performance overhead.


### 9.2. Discussion

The most striking feature of the `TMP_DF8` database is its rigid and consistent adherence to a **vertical partitioning** architecture. This characteristic is the root cause of all its major strengths and weaknesses. The schema is essentially a single, massive spreadsheet that has been sliced vertically into 27 thematic columns or groups of columns. The uniformity of the 5,050-row count across nearly every table is the clearest evidence of this. This design, inherited from its origins as a VAX "random access" file system, represents a data management paradigm that prioritizes organizational segmentation over relational integrity and query performance.

While this structure might have offered benefits in a legacy mainframe environment where data was accessed programmatically by file, it proves to be a significant liability in a modern analytical context that relies on the relational query model (SQL). The database's primary characteristic is therefore one of **structural fragmentation without true normalization**. It has the high table count and join complexity of a normalized system but lacks the enforced referential integrity and potential for complex relational modeling. This core characteristic directly leads to the observed performance penalties, the cumbersome requirement for external codebooks to interpret `smallint` codes, and the data integrity risks associated with sentinel values. The entire profile of this database is a testament to its legacy, serving as a clear case study in why architectural designs must align with their intended use case.


### 9.3. Conclusions & Implications for Phase 2 Redesign

Based on the comprehensive analysis of its structure, data quality, and performance, this report concludes that the `TMP_DF8` database, while a historically significant dataset, possesses an architecture that is fundamentally ill-suited for modern, efficient, and reliable data analysis. Its design choices, rooted in the technological constraints of a past computing era, create significant barriers to usability and performance that must be addressed in the Phase 2 redesign.

*   **Based on this analysis, what are the key strengths and weaknesses of this database's design?**
    *   **Strengths:**
        *   **Structural Consistency:** The database exhibits a highly consistent and predictable structure. The vertical partitioning is applied uniformly, and the use of the `ssn` as a universal key makes the logical relationships, though not formally enforced, easy to understand.
        *   **Data Completeness (Nominal):** From a purely technical standpoint (ignoring the sentinel value issue), the database is complete, with no `NULL` values that could complicate certain types of joins or software interactions.

    *   **Weaknesses:**
        *   **Query Performance Penalty:** The vertically partitioned design imposes a significant and measurable performance penalty (a 7-8x slowdown) on essential analytical queries that require joining data across tables.
        *   **Poor Usability:** The overwhelming reliance on `smallint` codes for categorical data makes the database opaque and difficult to use without constant reference to external documentation, hindering exploratory analysis.
        *   **Data Integrity Risks:** The use of sentinel values (`-1`, `-999`) instead of standard SQL `NULL`s creates a high risk of corrupting statistical calculations and producing erroneous analytical results.
        *   **Lack of Formal Integrity:** The absence of enforced foreign key constraints between the partitioned tables means the database lacks the built-in referential integrity of a true relational system.

*   **What specific aspects of this schema should be preserved, changed, or discarded in the final unified database?**
    *   **Discard:**
        *   **Vertical Partitioning:** The core architectural strategy of splitting a single logical entity across dozens of tables must be discarded entirely. All attributes for a single archaeological site (`ssn`) should be consolidated into a single record.
        *   **Sentinel Values for Missing Data:** The practice of using `-1` and `-999` to represent missingness must be completely abandoned. These values should be systematically and carefully converted to standard SQL `NULL`s.
        *   **Numeric Coding for Categorical Data:** The use of `smallint` codes for non-numeric, categorical data should be discarded.

    *   **Change:**
        *   **Data Types:** All columns containing categorical data should be changed from `smallint` to a descriptive `TEXT` or `VARCHAR` type. The integer codes should be replaced with their human-readable string descriptions (e.g., `1` becomes `"Ordinary"`).

    *   **Preserve:**
        *   **Core Data Content:** The actual data and the rich set of variables captured in `TMP_DF8` are invaluable. The content of the database—the artifact counts, site observations, and locational information—is the core asset that must be preserved and carried forward into the new, more efficient schema.
        *   **The `ssn` as a Primary Key:** The concept of the `ssn` as the unique identifier for each archaeological site is sound and should be preserved as the primary key in the new, unified table.
